# Ship & Oil Data Preprocessing Notebook
This notebook applies strict cleaning, cross-validation, and final validation rules.

This version is organized into multiple small cells for easier understanding. The logic and outputs are unchanged.

## Implemented Rules
- Handles mixed input date formats and converts all output dates to `dd-mm-yyyy`.
- Removes missing IDs and duplicates (`ship_id`, `oil_id`).
- Cleans numeric fields, removes invalid/negative values, and trims top/bottom 1% outliers.
- Standardizes text fields (trim + title case + name normalization).
- Fills missing `last_oil_type` with mode.
- Applies cross-validation between oil and ship datasets.
- Performs final validation (nulls, types, duplicates, negatives, logical consistency).
- Saves cleaned CSV outputs for allocation usage.

## 1) Import libraries and set display options
This cell imports all required libraries and configures pandas display settings.

In [1]:
import os
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

## 2) Define file paths
This cell defines raw/clean data locations and ensures the cleaned output directory exists.

In [2]:
DATA_DIR = os.path.join("..", "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
CLEAN_DIR = os.path.join(DATA_DIR, "cleaned")
os.makedirs(CLEAN_DIR, exist_ok=True)

OIL_RAW_PATH = os.path.join(RAW_DIR, "oil_data_raw.csv")
SHIP_RAW_PATH = os.path.join(RAW_DIR, "ship_data_raw.csv")
OIL_CLEAN_PATH = os.path.join(CLEAN_DIR, "oil_data_cleaned.csv")
SHIP_CLEAN_PATH = os.path.join(CLEAN_DIR, "ship_data_cleaned.csv")

## 3) Text standardization helpers
These helper functions normalize whitespace, title-case text, and map common abbreviations to standardized names.

In [3]:
def clean_spaces_and_title(value: str) -> str:
    if pd.isna(value):
        return np.nan
    cleaned = re.sub(r"\s+", " ", str(value)).strip()
    return cleaned.title()

def standardize_name(value: str) -> str:
    if pd.isna(value):
        return np.nan
    value = clean_spaces_and_title(value)
    replacements = {
        "Uae": "United Arab Emirates",
        "Usa": "United States",
        "Uk": "United Kingdom",
        "Ksa": "Saudi Arabia",
        "Nyc": "New York",
    }
    return replacements.get(value, value)

## 4) Date parsing and outlier helpers
This cell defines reusable helpers for date normalization and 1% outlier trimming.

In [4]:
def parse_to_ddmmyyyy(series: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(series, errors="coerce", dayfirst=True)
    return parsed.dt.strftime("%d-%m-%Y")

def remove_outliers_1pct(df: pd.DataFrame, column_name: str) -> pd.DataFrame:
    low = df[column_name].quantile(0.01)
    high = df[column_name].quantile(0.99)
    return df[(df[column_name] >= low) & (df[column_name] <= high)].copy()

## 5) Ship data cleaning
This function cleans ship IDs, capacity, dates, ports, and fills missing `last_oil_type` with mode.

In [5]:
def clean_ship_data(ship_df: pd.DataFrame) -> pd.DataFrame:
    df = ship_df.copy()

    # ship_id rules
    df["ship_id"] = df["ship_id"].astype("string").str.strip()
    df = df[df["ship_id"].notna() & (df["ship_id"] != "")].copy()
    df = df.drop_duplicates(subset=["ship_id"], keep="first")

    # capacity_mt rules
    df["capacity_mt"] = pd.to_numeric(df["capacity_mt"], errors="coerce")
    df = df[df["capacity_mt"].notna() & (df["capacity_mt"] > 0)].copy()
    if len(df) > 3:
        df = remove_outliers_1pct(df, "capacity_mt")

    # available_date rules
    parsed_dates = pd.to_datetime(df["available_date"], errors="coerce", dayfirst=True)
    df = df[parsed_dates.notna()].copy()
    df["available_date"] = parsed_dates.loc[df.index].dt.strftime("%d-%m-%Y")

    # available_port cleanup
    df["available_port"] = df["available_port"].apply(standardize_name)

    # last_oil_type rules
    df["last_oil_type"] = df["last_oil_type"].apply(clean_spaces_and_title)
    mode_series = df["last_oil_type"].dropna().mode()
    fill_mode = mode_series.iloc[0] if not mode_series.empty else "Unknown"
    df["last_oil_type"] = df["last_oil_type"].fillna(fill_mode)

    return df.reset_index(drop=True)

## 6) Oil data cleaning
This function cleans oil IDs, quantities, dates, and standardizes origin/destination fields.

In [6]:
def clean_oil_data(oil_df: pd.DataFrame) -> pd.DataFrame:
    df = oil_df.copy()

    # oil_id rules
    df["oil_id"] = df["oil_id"].astype("string").str.strip()
    df = df[df["oil_id"].notna() & (df["oil_id"] != "")].copy()
    df = df.drop_duplicates(subset=["oil_id"], keep="first")

    # quantity rules (support both names)
    quantity_col = "quantity_mt" if "quantity_mt" in df.columns else "quantity_metric_tons"
    df[quantity_col] = pd.to_numeric(df[quantity_col], errors="coerce")
    df = df[df[quantity_col].notna() & (df[quantity_col] > 0)].copy()
    if len(df) > 3:
        df = remove_outliers_1pct(df, quantity_col)

    # delivery_deadline rules
    parsed_dates = pd.to_datetime(df["delivery_deadline"], errors="coerce", dayfirst=True)
    df = df[parsed_dates.notna()].copy()
    df["delivery_deadline"] = parsed_dates.loc[df.index].dt.strftime("%d-%m-%Y")

    # origin/destination cleanup + standardization
    df["origin_country"] = df["origin_country"].apply(standardize_name)
    df["destination_port"] = df["destination_port"].apply(standardize_name)

    # If input had quantity_metric_tons, rename to quantity_mt for downstream consistency
    if quantity_col == "quantity_metric_tons":
        df = df.rename(columns={"quantity_metric_tons": "quantity_mt"})

    return df.reset_index(drop=True)

## 7) Cross-validate oil and ship compatibility
This function keeps only rows that can potentially match based on capacity, date, and optional port match.

In [7]:
def cross_validate(oil_df: pd.DataFrame, ship_df: pd.DataFrame, require_port_match: bool = True) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    oil = oil_df.copy()
    ship = ship_df.copy()

    oil_deadline = pd.to_datetime(oil["delivery_deadline"], format="%d-%m-%Y", errors="coerce")
    ship_available = pd.to_datetime(ship["available_date"], format="%d-%m-%Y", errors="coerce")

    feasible_counts = []
    for _, oil_row in oil.iterrows():
        mask = (
            (ship["capacity_mt"] >= oil_row["quantity_mt"])
            & (ship_available <= pd.to_datetime(oil_row["delivery_deadline"], format="%d-%m-%Y", errors="coerce"))
        )
        if require_port_match:
            mask = mask & (ship["available_port"] == oil_row["destination_port"])

        feasible_counts.append(int(mask.sum()))

    oil["feasible_ship_count"] = feasible_counts
    oil_valid = oil[oil["feasible_ship_count"] > 0].drop(columns=["feasible_ship_count"]).copy()

    # Keep ships that can serve at least one valid oil row
    if oil_valid.empty:
        ship_valid = ship.iloc[0:0].copy()
    else:
        valid_mask = pd.Series(False, index=ship.index)
        oil_valid_deadline = pd.to_datetime(oil_valid["delivery_deadline"], format="%d-%m-%Y", errors="coerce")
        for _, ship_row in ship.iterrows():
            mask = (
                (oil_valid["quantity_mt"] <= ship_row["capacity_mt"])
                & (pd.to_datetime(ship_row["available_date"], format="%d-%m-%Y", errors="coerce") <= oil_valid_deadline)
            )
            if require_port_match:
                mask = mask & (oil_valid["destination_port"] == ship_row["available_port"])
            if mask.any():
                valid_mask.loc[ship_row.name] = True
        ship_valid = ship[valid_mask].copy()

    report = {
        "oil_rows_before_cross_validation": int(len(oil_df)),
        "oil_rows_after_cross_validation": int(len(oil_valid)),
        "ship_rows_before_cross_validation": int(len(ship_df)),
        "ship_rows_after_cross_validation": int(len(ship_valid)),
        "oil_rows_removed_by_cross_validation": int(len(oil_df) - len(oil_valid)),
        "ship_rows_removed_by_cross_validation": int(len(ship_df) - len(ship_valid)),
    }
    return oil_valid.reset_index(drop=True), ship_valid.reset_index(drop=True), report

## 8) Final validation checks
This function validates nulls, uniqueness, positivity, and date format compliance after cleaning/cross-validation.

In [8]:
def final_validation(oil_df: pd.DataFrame, ship_df: pd.DataFrame) -> dict:
    checks = {}

    checks["oil_no_nulls"] = bool(not oil_df.isna().any().any())
    checks["ship_no_nulls"] = bool(not ship_df.isna().any().any())
    checks["oil_unique_oil_id"] = bool(not oil_df["oil_id"].duplicated().any())
    checks["ship_unique_ship_id"] = bool(not ship_df["ship_id"].duplicated().any())
    checks["oil_quantity_positive"] = bool((oil_df["quantity_mt"] > 0).all())
    checks["ship_capacity_positive"] = bool((ship_df["capacity_mt"] > 0).all())

    oil_dates = pd.to_datetime(oil_df["delivery_deadline"], format="%d-%m-%Y", errors="coerce")
    ship_dates = pd.to_datetime(ship_df["available_date"], format="%d-%m-%Y", errors="coerce")
    checks["oil_dates_valid_ddmmyyyy"] = bool(oil_dates.notna().all())
    checks["ship_dates_valid_ddmmyyyy"] = bool(ship_dates.notna().all())

    checks["all_checks_passed"] = bool(all(checks.values()))
    return checks

## 9) Execute the full pipeline and print summary
This cell loads raw files, runs cleaning + cross-validation + final validation, saves cleaned CSVs, and displays results.

In [9]:
# Load raw files
oil_raw = pd.read_csv(OIL_RAW_PATH)
ship_raw = pd.read_csv(SHIP_RAW_PATH)

# Clean independently
oil_clean = clean_oil_data(oil_raw)
ship_clean = clean_ship_data(ship_raw)

# Cross-validation
oil_final, ship_final, cross_report = cross_validate(
    oil_clean,
    ship_clean,
    require_port_match=True,
    # Set to False if your allocation logic does not require strict port match
 )

# Final validation
validation_report = final_validation(oil_final, ship_final)

# Save cleaned outputs
oil_final.to_csv(OIL_CLEAN_PATH, index=False)
ship_final.to_csv(SHIP_CLEAN_PATH, index=False)

# Summary statistics
summary = {
    "ship_rows_raw": int(len(ship_raw)),
    "ship_rows_cleaned": int(len(ship_final)),
    "oil_rows_raw": int(len(oil_raw)),
    "oil_rows_cleaned": int(len(oil_final)),
    "cross_validation": cross_report,
    "final_validation": validation_report,
    "saved_ship_csv": SHIP_CLEAN_PATH,
    "saved_oil_csv": OIL_CLEAN_PATH,
}

print("=== SUMMARY ===")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\n=== CLEANED SHIP SAMPLE ===")
display(ship_final.head())

print("\n=== CLEANED OIL SAMPLE ===")
display(oil_final.head())

=== SUMMARY ===
ship_rows_raw: 20
ship_rows_cleaned: 0
oil_rows_raw: 20
oil_rows_cleaned: 0
cross_validation: {'oil_rows_before_cross_validation': 18, 'oil_rows_after_cross_validation': 0, 'ship_rows_before_cross_validation': 18, 'ship_rows_after_cross_validation': 0, 'oil_rows_removed_by_cross_validation': 18, 'ship_rows_removed_by_cross_validation': 18}
final_validation: {'oil_no_nulls': True, 'ship_no_nulls': True, 'oil_unique_oil_id': True, 'ship_unique_ship_id': True, 'oil_quantity_positive': True, 'ship_capacity_positive': True, 'oil_dates_valid_ddmmyyyy': True, 'ship_dates_valid_ddmmyyyy': True, 'all_checks_passed': True}
saved_ship_csv: ..\data\cleaned\ship_data_cleaned.csv
saved_oil_csv: ..\data\cleaned\oil_data_cleaned.csv

=== CLEANED SHIP SAMPLE ===


,ship_id,capacity_mt,last_oil_type,available_date,available_port



=== CLEANED OIL SAMPLE ===


,oil_id,oil_type,delivery_deadline,origin_country,destination_port,quantity_mt,origin_port


In [10]:
import os
import re
import numpy as np
import pandas as pd

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()).lower() == "notebooks" else os.getcwd()
RAW_DIR = os.path.join(ROOT_DIR, "data", "raw")
CLEAN_DIR = os.path.join(ROOT_DIR, "data", "cleaned")
os.makedirs(CLEAN_DIR, exist_ok=True)

OIL_RAW = os.path.join(RAW_DIR, "oil_data_raw.csv")
SHIP_RAW = os.path.join(RAW_DIR, "ship_data_raw.csv")
OIL_CLEAN = os.path.join(CLEAN_DIR, "oil_data_cleaned.csv")
SHIP_CLEAN = os.path.join(CLEAN_DIR, "ship_data_cleaned.csv")


def clean_spaces_and_title(value: str):
    if pd.isna(value):
        return np.nan
    cleaned = re.sub(r"\s+", " ", str(value)).strip()
    return cleaned.title()


def standardize_name(value: str):
    if pd.isna(value):
        return np.nan
    value = clean_spaces_and_title(value)
    replacements = {
        "Uae": "United Arab Emirates",
        "Usa": "United States",
        "Uk": "United Kingdom",
        "Ksa": "Saudi Arabia",
        "Nyc": "New York",
    }
    return replacements.get(value, value)


def remove_outliers_1pct(df: pd.DataFrame, column_name: str) -> pd.DataFrame:
    low = df[column_name].quantile(0.01)
    high = df[column_name].quantile(0.99)
    return df[(df[column_name] >= low) & (df[column_name] <= high)].copy()


def clean_ship_data(ship_df: pd.DataFrame) -> pd.DataFrame:
    df = ship_df.copy()
    df["ship_id"] = df["ship_id"].astype("string").str.strip()
    df = df[df["ship_id"].notna() & (df["ship_id"] != "")].copy()
    df = df.drop_duplicates(subset=["ship_id"], keep="first")

    df["capacity_mt"] = pd.to_numeric(df["capacity_mt"], errors="coerce")
    df = df[df["capacity_mt"].notna() & (df["capacity_mt"] > 0)].copy()
    if len(df) > 3:
        df = remove_outliers_1pct(df, "capacity_mt")

    parsed_dates = pd.to_datetime(df["available_date"], errors="coerce", dayfirst=True)
    df = df[parsed_dates.notna()].copy()
    df["available_date"] = parsed_dates.loc[df.index].dt.strftime("%d-%m-%Y")

    df["available_port"] = df["available_port"].apply(standardize_name)
    df["last_oil_type"] = df["last_oil_type"].apply(clean_spaces_and_title)

    mode_series = df["last_oil_type"].dropna().mode()
    fill_mode = mode_series.iloc[0] if not mode_series.empty else "Unknown"
    df["last_oil_type"] = df["last_oil_type"].fillna(fill_mode)

    df["available_port"] = df["available_port"].astype("string").str.strip()
    df = df[
        df["available_port"].notna()
        & (df["available_port"] != "")
        & (df["available_port"].str.lower() != "nan")
    ].copy()

    return df.reset_index(drop=True)


def clean_oil_data(oil_df: pd.DataFrame) -> pd.DataFrame:
    df = oil_df.copy()
    df["oil_id"] = df["oil_id"].astype("string").str.strip()
    df = df[df["oil_id"].notna() & (df["oil_id"] != "")].copy()
    df = df.drop_duplicates(subset=["oil_id"], keep="first")

    quantity_col = "quantity_mt" if "quantity_mt" in df.columns else "quantity_metric_tons"
    df[quantity_col] = pd.to_numeric(df[quantity_col], errors="coerce")
    df = df[df[quantity_col].notna() & (df[quantity_col] > 0)].copy()
    if len(df) > 3:
        df = remove_outliers_1pct(df, quantity_col)

    parsed_dates = pd.to_datetime(df["delivery_deadline"], errors="coerce", dayfirst=True)
    df = df[parsed_dates.notna()].copy()
    df["delivery_deadline"] = parsed_dates.loc[df.index].dt.strftime("%d-%m-%Y")

    df["oil_type"] = df["oil_type"].apply(clean_spaces_and_title)
    df["origin_country"] = df["origin_country"].apply(standardize_name)
    df["destination_port"] = df["destination_port"].apply(standardize_name)

    for column in ["oil_type", "origin_country", "destination_port"]:
        df[column] = df[column].astype("string").str.strip()

    df = df[
        df["oil_type"].notna()
        & (df["oil_type"] != "")
        & (df["oil_type"].str.lower() != "nan")
        & df["origin_country"].notna()
        & (df["origin_country"] != "")
        & (df["origin_country"].str.lower() != "nan")
        & df["destination_port"].notna()
        & (df["destination_port"] != "")
        & (df["destination_port"].str.lower() != "nan")
    ].copy()

    if quantity_col == "quantity_metric_tons":
        df = df.rename(columns={"quantity_metric_tons": "quantity_mt"})

    return df.reset_index(drop=True)


oil_df = pd.read_csv(OIL_RAW)
ship_df = pd.read_csv(SHIP_RAW)

cleaned_oil = clean_oil_data(oil_df)
cleaned_ship = clean_ship_data(ship_df)

cleaned_oil.to_csv(OIL_CLEAN, index=False)
cleaned_ship.to_csv(SHIP_CLEAN, index=False)

print(f"Cleaned oil rows: {len(cleaned_oil)}")
print(f"Cleaned ship rows: {len(cleaned_ship)}")
print(f"Wrote: {OIL_CLEAN}")
print(f"Wrote: {SHIP_CLEAN}")

Cleaned oil rows: 16
Cleaned ship rows: 17
Wrote: C:\Users\gangt\OneDrive\Desktop\oil_ship_assignment\data\cleaned\oil_data_cleaned.csv
Wrote: C:\Users\gangt\OneDrive\Desktop\oil_ship_assignment\data\cleaned\ship_data_cleaned.csv
